# Visualização da Informação — Cotação USD/BRL (2010–2019)

**Dataset real:** cotação diária do dólar americano em reais (USD/BRL), 2.608 registros de 01/01/2010 a 31/12/2019 (`data/USD_BRL_hist.csv`). Link do dataset: _preencher com a URL pública da fonte_.

Três gráficos de **unidades diferentes** da disciplina, todos sobre o mesmo dataset e encadeados em uma única pergunta: *como o dólar se comportou na década e quando ficou mais caro?*

| Unidade | Técnica | Papel na narrativa |
|---|---|---|
| Estatística descritiva | Barras | Resumo anual (média) |
| Temporal | Linhas | Evolução diária e tendência |
| Hierárquica | Sunburst | Ano → trimestre → mês, colorido pela média |

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('data/USD_BRL_hist.csv')
df['Data'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')
df = df.sort_values('Data').reset_index(drop=True)

# Colunas derivadas: dão a estrutura hierárquica ano > trimestre > mês
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
df['ano'] = df['Data'].dt.year.astype(str)
df['trimestre'] = 'T' + df['Data'].dt.quarter.astype(str)
df['mes'] = df['Data'].dt.month.map(lambda m: meses[m - 1])

print(df.shape, df['Data'].min().date(), '->', df['Data'].max().date())
print('Nulos:', int(df['USD_BRL'].isna().sum()))
df.head()

## Plot 1 — Barras (Estatística descritiva)
A média anual é uma grandeza resumida por categoria discreta (ano), a natureza ideal para barras: comprimento comparável a partir de uma base comum.

In [ ]:
anual = df.groupby('ano')['USD_BRL'].agg(media='mean', minimo='min', maximo='max').reset_index()

fig1 = px.bar(
    anual, x='ano', y='media', text=anual['media'].round(2),
    hover_data={'minimo': ':.2f', 'maximo': ':.2f', 'media': ':.2f'},
    labels={'ano': 'Ano', 'media': 'Média anual (R$ por US$)'},
    title='Cotação média anual do dólar (2010–2019)',
)
fig1.update_traces(marker_color='#2a6f97', textposition='outside')
fig1.update_layout(yaxis_range=[0, anual['media'].max() * 1.15])
fig1.show()

## Plot 2 — Linhas (Temporal)
Série temporal contínua: a linha preserva a ordem e a continuidade dos dias. A média móvel de 60 dias separa a tendência do ruído diário.

In [ ]:
df['media_movel_60d'] = df['USD_BRL'].rolling(60, min_periods=1).mean()

fig2 = px.line(
    df, x='Data', y=['USD_BRL', 'media_movel_60d'],
    labels={'Data': 'Data', 'value': 'R$ por US$', 'variable': 'Série'},
    title='Evolução diária do dólar e média móvel de 60 dias',
)
fig2.data[0].name = 'Cotação diária'
fig2.data[0].line.update(color='#a9c6dc', width=1)
fig2.data[1].name = 'Média móvel 60 dias'
fig2.data[1].line.update(color='#c1121f', width=2.5)
fig2.update_xaxes(rangeslider_visible=True)
fig2.show()

## Plot 3 — Sunburst (Hierárquica)
O tempo tem hierarquia natural (ano → trimestre → mês). O sunburst mostra essa estrutura; a cor codifica a cotação média de cada fatia e o ângulo, o número de dias, o que revela quais períodos concentraram dólar mais caro.

In [ ]:
fig3 = px.sunburst(
    df, path=['ano', 'trimestre', 'mes'],
    color='USD_BRL', color_continuous_scale='YlOrRd',
    labels={'USD_BRL': 'Cotação média (R$)'},
    title='Cotação média do dólar por ano > trimestre > mês',
)
fig3.update_layout(width=750, height=750)
fig3.show()

## Conclusão
- As **barras** mostram a escalada da média anual: de R$ 1,76 (2010) a R$ 3,95 (2019), com salto de 2014 (R$ 2,35) para 2015 (R$ 3,34).
- As **linhas** mostram quando isso ocorreu, a volatilidade diária e a máxima da década (R$ 4,26 em 27/11/2019).
- O **sunburst** detalha, dentro de cada ano, quais trimestres e meses foram mais caros.

_Revisar os comentários acima contra os gráficos gerados antes de gravar o vídeo._